# Registered Users — Bronze to Silver

This notebook transforms raw auction registration records from the **bronze** layer into a curated **silver** Delta table for downstream reporting and analytics.

It works at the **registered user level**, meaning each row represents a bidder registration tied to an auction. The notebook standardizes names, parses auction dates, and loads the cleaned records into `company_db.silver.registeredusers`.

## What this notebook is doing

This sheet provides a breakdown of auction registrants, including:
* which auction they registered for
* the bidder ID tied to the registration
* the registrant's full name and contact details
* billing and shipping information captured at registration
* whether the registration was approved or auto-approved
* registration and approval timestamps

## Main transformations

1. Reads raw registration-level records from `company_db.bronze.registeredusers`.
2. Builds a cleaned `full_name` from first and last name.
3. Renames `bidder_number` to `bidder_id` for analytics-friendly naming.
4. Splits the auction date range into `start_date` and `end_date`.
5. Casts the parsed date fields to proper Spark `DATE` values.
6. Creates the target Delta table in the silver layer.
7. Registers the transformed DataFrame as a temporary view.
8. Merges transformed records into silver so existing registrations are updated and new registrations are inserted.

## Source and target

* **Source:** `company_db.bronze.registeredusers`
* **Target:** `company_db.silver.registeredusers`
* **Storage location:** `abfss://auction@company.dfs.core.windows.net/silver/registeredusers`

## Data dictionary

| Column | Meaning |
| --- | --- |
| `platform` | Source platform where the bidder registration was captured. |
| `auction_name` | Name of the auction tied to the registration. |
| `auction_date` | Original auction date range text from the source system. |
| `start_date` | Parsed auction start date from `auction_date`. |
| `end_date` | Parsed auction end date from `auction_date`. |
| `bidder_id` | Bidder number assigned to the registered user. |
| `first_name` | Registrant first name. |
| `last_name` | Registrant last name. |
| `full_name` | Proper-cased full name built from first and last name. |
| `phone_number` | Main phone number for the registrant. |
| `email` | Email address used for registration. |
| `billing_company` | Billing company name on the registration record. |
| `billing_first_name` | Billing contact first name. |
| `billing_last_name` | Billing contact last name. |
| `billing_phone_number` | Billing contact phone number. |
| `billing_address_1` | Primary billing street address. |
| `billing_address_2` | Secondary billing address line. |
| `billing_city` | Billing city. |
| `billing_state_province` | Billing state or province. |
| `billing_zip_postal_code` | Billing ZIP or postal code. |
| `billing_country` | Billing country. |
| `shipping_company` | Shipping company name on the registration record. |
| `shipping_first_name` | Shipping contact first name. |
| `shipping_last_name` | Shipping contact last name. |
| `shipping_phone_number` | Shipping contact phone number. |
| `shipping_address_1` | Primary shipping street address. |
| `shipping_address_2` | Secondary shipping address line. |
| `shipping_city` | Shipping city. |
| `shipping_state_province` | Shipping state or province. |
| `shipping_zip_postal_code` | Shipping ZIP or postal code. |
| `shipping_country` | Shipping country. |
| `registration_status` | Status of the bidder registration, such as pending or approved. |
| `auto_approved` | Indicates whether the registration was auto-approved. |
| `registration_datetime` | Timestamp when the user registered for the auction. |
| `date_approved` | Timestamp when the registration was approved. |
| `path` | Source file or ingestion path used for traceability. |
| `ingestion_timestamp` | Timestamp when the raw record was ingested. |
| `created_at` | Timestamp when the record was first inserted into silver. |
| `updated_at` | Timestamp when the record was last updated in silver. |

## Business interpretation

* A row represents one bidder registration for a specific auction.
* `registration_status` shows the operational state of that registration.
* `auto_approved` helps distinguish registrations approved automatically from those reviewed manually.
* The silver table supports operational reporting, bidder analysis, registration trends, and downstream joins to auction and bidding data.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import PySpark SQL functions used for column transformations, casting, parsing, and conditional logic.
from pyspark.sql import functions as F
from pyspark.sql.functions import split

In [ ]:
# Read the raw registered users table from the bronze layer.
# This DataFrame is the source for all registration-level cleaning and business transformations.
registeredusers = spark.table(f'{catalog}.bronze.registeredusers').filter(F.col('batch_id')==batch_id)

In [ ]:
from src.transformations.registered_users import transform_registered_users

# Apply transformation function
registeredusers = transform_registered_users(registeredusers)

# Read Bronze users for enrichment
users = spark.table(
    f"{catalog}.bronze.users"
).filter(F.col("batch_id") == batch_id)

users = (
    users
    .withColumnRenamed("user_#", "user_id")
    .withColumnRenamed("email_address", "email")
    .select("email", "user_id")
)

# Join and deduplicate
registeredusers = (
    registeredusers
    .join(users, on="email", how="inner")
    .dropDuplicates(["email"])
)

In [ ]:
auction_sales_report = spark.table(f'{catalog}.silver.auction_sales_report').filter(F.col("batch_id") == batch_id)
auction_sales_report = auction_sales_report.select('start_date','auction_id')
registeredusers = registeredusers.join(auction_sales_report,'start_date','left')

In [ ]:
# Create or replace the silver Delta table that stores curated auction registration records.
# This schema supports reporting on registrants, approval status, and contact details.

spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.registeredusers (
    auction_id Int,
    platform STRING,
    auction_name STRING,
    auction_date STRING,
    start_date DATE,
    end_date DATE,
    bidder_id INT,
    user_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    phone_number STRING,
    email STRING,
    billing_company STRING,
    billing_first_name STRING,
    billing_last_name STRING,
    billing_phone_number STRING,
    billing_address_1 STRING,
    billing_address_2 STRING,
    billing_city STRING,
    billing_state_province STRING,
    billing_zip_postal_code STRING,
    billing_country STRING,
    shipping_company STRING,
    shipping_first_name STRING,
    shipping_last_name STRING,
    shipping_phone_number STRING,
    shipping_address_1 STRING,
    shipping_address_2 STRING,
    shipping_city STRING,
    shipping_state_province STRING,
    shipping_zip_postal_code STRING,
    shipping_country STRING,
    registration_status STRING,
    auto_approved STRING,
    registration_datetime TIMESTAMP,
    date_approved TIMESTAMP,
    path STRING,
    ingestion_timestamp TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    batch_id STRING
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/registeredusers';""")

In [ ]:
# Register the transformed DataFrame as a temporary view so it can be used by the SQL MERGE statement.
registeredusers.createOrReplaceTempView('registeredusers_view')

In [ ]:
# Upsert transformed registration records into the silver registered users table.
# Existing rows are updated by email and auction start date, and new rows are inserted.

spark.sql(
f"""
MERGE INTO {catalog}.silver.registeredusers AS tgt
USING registeredusers_view AS src
ON tgt.email = src.email
AND
tgt.auction_id = src.auction_id
AND
tgt.start_date = src.start_date

WHEN MATCHED
AND tgt.batch_id >= src.batch_id  
 THEN UPDATE SET
    tgt.auction_id = src.auction_id,
    tgt.auction_name = src.auction_name,
    tgt.auction_date = src.auction_date,
    tgt.platform = src.platform,
    tgt.start_date = src.start_date,
    tgt.end_date = src.end_date,
    tgt.bidder_id = src.bidder_id,
    tgt.user_id = src.user_id ,
    tgt.first_name = src.first_name,
    tgt.last_name = src.last_name,
    tgt.full_name = src.full_name,
    tgt.phone_number = src.phone_number,
    tgt.email = src.email,
    tgt.billing_company = src.billing_company,
    tgt.billing_first_name = src.billing_first_name,
    tgt.billing_last_name = src.billing_last_name,
    tgt.billing_phone_number = src.billing_phone_number,
    tgt.billing_address_1 = src.billing_address_1,
    tgt.billing_address_2 = src.billing_address_2,
    tgt.billing_city = src.billing_city,
    tgt.billing_state_province = src.billing_state_province,
    tgt.billing_zip_postal_code = src.billing_zip_postal_code,
    tgt.billing_country = src.billing_country,
    tgt.shipping_company = src.shipping_company,
    tgt.shipping_first_name = src.shipping_first_name,
    tgt.shipping_last_name = src.shipping_last_name,
    tgt.shipping_phone_number = src.shipping_phone_number,
    tgt.shipping_address_1 = src.shipping_address_1,
    tgt.shipping_address_2 = src.shipping_address_2,
    tgt.shipping_city = src.shipping_city,
    tgt.shipping_state_province = src.shipping_state_province,
    tgt.shipping_zip_postal_code = src.shipping_zip_postal_code,
    tgt.shipping_country = src.shipping_country,
    tgt.registration_status = src.registration_status,
    tgt.auto_approved = src.auto_approved,
    tgt.registration_datetime = src.registration_datetime,
    tgt.date_approved = src.date_approved,
    tgt.path = src.path,
    tgt.ingestion_timestamp = src.ingestion_timestamp,
    tgt.updated_at = current_timestamp(),
    tgt.batch_id = src.batch_id

WHEN NOT MATCHED THEN INSERT (
    auction_id,
    platform,
    auction_name,
    auction_date,
    start_date,
    end_date,
    bidder_id,
    user_id,
    first_name,
    last_name,
    full_name,
    phone_number,
    email,
    billing_company,
    billing_first_name,
    billing_last_name,
    billing_phone_number,
    billing_address_1,
    billing_address_2,
    billing_city,
    billing_state_province,
    billing_zip_postal_code,
    billing_country,
    shipping_company,
    shipping_first_name,
    shipping_last_name,
    shipping_phone_number,
    shipping_address_1,
    shipping_address_2,
    shipping_city,
    shipping_state_province,
    shipping_zip_postal_code,
    shipping_country,
    registration_status,
    auto_approved,
    registration_datetime,
    date_approved,
    path,
    ingestion_timestamp,
    created_at,
    updated_at,
    batch_id
)
VALUES (
    src.auction_id,
    src.platform,
    src.auction_name,
    src.auction_date,
    src.start_date,
    src.end_date,
    src.bidder_id,
    src.user_id,
    src.first_name,
    src.last_name,
    src.full_name,
    src.phone_number,
    src.email,
    src.billing_company,
    src.billing_first_name,
    src.billing_last_name,
    src.billing_phone_number,
    src.billing_address_1,
    src.billing_address_2,
    src.billing_city,
    src.billing_state_province,
    src.billing_zip_postal_code,
    src.billing_country,
    src.shipping_company,
    src.shipping_first_name,
    src.shipping_last_name,
    src.shipping_phone_number,
    src.shipping_address_1,
    src.shipping_address_2,
    src.shipping_city,
    src.shipping_state_province,
    src.shipping_zip_postal_code,
    src.shipping_country,
    src.registration_status,
    src.auto_approved,
    src.registration_datetime,
    src.date_approved,
    src.path,
    src.ingestion_timestamp,
    current_timestamp(),
    current_timestamp(),
    src.batch_id
);""")

## Analyst notes

Use this table when you need registration-level auction reporting such as:
* registrations by auction
* approved vs pending registration counts
* auto-approved registration tracking
* bidder contact and location analysis
* auction registration trend reporting
* downstream joins to bidding or auction participation tables

Because the notebook stores one row per bidder registration, it is well suited for operational reporting, analyst review, and silver-to-gold aggregations.

## Business usage notes

This silver table is the curated registration layer for auction participants.

Use it to answer questions such as:
* how many users registered for each auction
* which registrations were approved or auto-approved
* how registrant geography is distributed across billing or shipping countries
* which bidders can be matched to downstream bidding activity by email or bidder ID

This table is best treated as a registration fact table at the bidder-auction level.